In [1]:
# (optional) Install required packages
# !pip3 install seaborn
# !pip3 install pandas
# !pip3 install scikit-learn

Import the required libraries

In [2]:
import prepro_pipelines as pp
import model_pipelines as mp

Define the variables

In [3]:
DATA_PATH = "./diabetic_data.csv"
TARGET = "readmitted"
IDENTIFIERS = ["encounter_id", "patient_nbr"]
COLUMNS_TO_DROP = [
        "weight",
        "admission_source_id",
        "payer_code",
        "medical_specialty",
        "repaglinide",
        "nateglinide",
        "chlorpropamide",
        "acetohexamide",
        "tolbutamide",
        "acarbose",
        "miglitol",
        "troglitazone",
        "tolazamide",
        "examide",
        "citoglipton",
        "glyburide-metformin",
        "glipizide-metformin",
        "glimepiride-pioglitazone",
        "metformin-rosiglitazone",
        "metformin-pioglitazone",
    ]

Load the data

In [4]:
data = pp.load_data(DATA_PATH)

=== Loading data from ./diabetic_data.csv ===
Data loaded with shape: (101766, 50)


Preprocess the data

In [5]:
data = (
    data
    .pipe(pp.number_of_encounters)
    # Keep only the first encounter for each patient
    .pipe(pp.keep_first_encounter)
    # Drop columns with high missing values and unuseful columns
    .pipe(pp.drop_columns, COLUMNS_TO_DROP)
    # Drop rows with invalid gender
    .pipe(pp.drop_row_with_value, "gender", "Unknown/Invalid")
    # Drop rows with expired patients
    .pipe(pp.drop_row_with_value, "discharge_disposition_id", 11)
    .pipe(pp.drop_row_with_value, "discharge_disposition_id", 19)
    .pipe(pp.drop_row_with_value, "discharge_disposition_id", 20)
    .pipe(pp.drop_row_with_value, "discharge_disposition_id", 21)
    # Replace missing values in categorical columns with "None"
    .pipe(pp.replace_missing, "max_glu_serum", "None")
    .pipe(pp.replace_missing, "A1Cresult", "None")
    # Coarse class the diagnosis codes into broader categories
    .pipe(pp.coarse_class_diagnosis, "diag_1")
    .pipe(pp.coarse_class_diagnosis, "diag_2")
    .pipe(pp.coarse_class_diagnosis, "diag_3")
    # Remap unknown race to "Other"
    .pipe(pp.remap_column, "race", {"?": "Other"})
    # Remap medication changes for "Up" and "Down" to "Change"
    .pipe(pp.remap_column, "insulin", {"Up": "Change", "Down": "Change"})
    .pipe(pp.remap_column, "metformin", {"Up": "Change", "Down": "Change"})
    .pipe(pp.remap_column, "glimepiride", {"Up": "Change", "Down": "Change"})
    .pipe(pp.remap_column, "glipizide", {"Up": "Change", "Down": "Change"})
    .pipe(pp.remap_column, "glyburide", {"Up": "Change", "Down": "Change"})
    .pipe(pp.remap_column, "pioglitazone", {"Up": "Change", "Down": "Change"})
    .pipe(pp.remap_column, "rosiglitazone", {"Up": "Change", "Down": "Change"})
    # Group unknown admission types into 0 for "Other", shift 7 ("Trauma Center") to 5 
    .pipe(pp.remap_column, "admission_type_id", {5: 0, 6: 0, 8: 0, 7: 5})
    # Binarize the target variable: 0 for "NO", 0 for ">30" and 1 for "<30"
    .pipe(pp.remap_column, "readmitted", {"NO": 0, ">30": 0, "<30": 1})
    # Group unknown discharge types into 0 for "Other", shift values
    .pipe(
        pp.remap_column,
        "discharge_disposition_id",
        {
            18: 0,
            25: 0,
            26: 0,
            12: 11,
            13: 12,
            14: 13,
            15: 14,
            16: 15,
            17: 16,
            22: 17,
            23: 18,
            24: 19,
            30: 20,
            27: 21,
            28: 22,
            29: 23,
        },
    )
    .pipe(pp.polynomial_feature_expansion, 
                [
                    "time_in_hospital",
                    "num_lab_procedures",
                    "num_procedures",
                    "num_medications",
                    "number_outpatient",
                    "number_emergency",
                    "number_inpatient",
                    "number_diagnoses",
                ],
                degree=2
    )
    .pipe(pp.polynomial_feature_expansion_categorical, [
        "metformin",
        "glimepiride",
        "glipizide",
        "glyburide",
        "pioglitazone",
        "rosiglitazone",
    ], 2)
)

=== Creating 'num_encounters' feature ===
'num_encounters' feature created.
=== Keeping only the first encounter for each patient ===
Initial shape: (101766, 51), after keeping first encounter: (71518, 51)
=== Dropping columns ===
Remaining columns:
    • encounter_id
    • patient_nbr
    • race
    • gender
    • age
    • admission_type_id
    • discharge_disposition_id
    • time_in_hospital
    • num_lab_procedures
    • num_procedures
    • num_medications
    • number_outpatient
    • number_emergency
    • number_inpatient
    • diag_1
    • diag_2
    • diag_3
    • number_diagnoses
    • max_glu_serum
    • A1Cresult
    • metformin
    • glimepiride
    • glipizide
    • glyburide
    • pioglitazone
    • rosiglitazone
    • insulin
    • change
    • diabetesMed
    • readmitted
    • num_encounters
=== Dropping rows with value 'Unknown/Invalid' in column: gender ===
Dropped 3 rows with value 'Unknown/Invalid' in column 'gender'
=== Dropping rows with value '11' in column: 

/home/weishengtoh/github/5006/Team1_IT5006_Healthcare_Analytics_AY2526/notebooks/milestone2_toh/prepro_pipelines.py:232: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data.replace({column: mapping}, inplace=True)


Created new categorical interaction feature: metformin glimepiride
Created new categorical interaction feature: metformin glipizide
Created new categorical interaction feature: metformin glyburide
Created new categorical interaction feature: metformin pioglitazone
Created new categorical interaction feature: metformin rosiglitazone
Created new categorical interaction feature: glimepiride glipizide
Created new categorical interaction feature: glimepiride glyburide
Created new categorical interaction feature: glimepiride pioglitazone
Created new categorical interaction feature: glimepiride rosiglitazone
Created new categorical interaction feature: glipizide glyburide
Created new categorical interaction feature: glipizide pioglitazone
Created new categorical interaction feature: glipizide rosiglitazone
Created new categorical interaction feature: glyburide pioglitazone
Created new categorical interaction feature: glyburide rosiglitazone
Created new categorical interaction feature: pioglit

Save preprocessed data

In [6]:
data.pipe(pp.to_csv, "cj_data.csv")

=== Saving data to cj_data.csv ===
Data saved to cj_data.csv | shape: (70431, 74)


,encounter_id,patient_nbr,race,gender,age,admission_type_id,discharge_disposition_id,diag_1,diag_2,diag_3,...,glimepiride glipizide,glimepiride glyburide,glimepiride pioglitazone,glimepiride rosiglitazone,glipizide glyburide,glipizide pioglitazone,glipizide rosiglitazone,glyburide pioglitazone,glyburide rosiglitazone,pioglitazone rosiglitazone
0,24437208,135,Caucasian,Female,[50-60),2,1,Diseases of the circulatory system,Injury and poisoning,Diseases of the digestive system,...,No No,No Change,No No,No No,No Change,No No,No No,Change No,Change No,No No
1,29758806,378,Caucasian,Female,[50-60),3,1,Diseases of the musculoskeletal system and con...,Mental disorders,"Endocrine, nutritional and metabolic diseases,...",...,No No,No No,No No,No No,No No,No No,No No,No No,No No,No No
2,189899286,729,Caucasian,Female,[80-90),1,3,Injury and poisoning,Diseases of the respiratory system,External causes of injury,...,No No,No No,No No,No No,No No,No No,No No,No No,No No,No No
3,64331490,774,Caucasian,Female,[80-90),1,1,"Endocrine, nutritional and metabolic diseases,...",Diseases of the circulatory system,Diseases of the circulatory system,...,No No,No Steady,No No,No No,No Steady,No No,No No,Steady No,Steady No,No No
4,14824206,927,AfricanAmerican,Female,[30-40),1,1,Diseases of the genitourinary system,Neoplasms,"Endocrine, nutritional and metabolic diseases,...",...,Steady No,Steady No,Steady No,Steady No,No No,No No,No No,No No,No No,No No
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
70426,418513058,189351095,Caucasian,Female,[80-90),1,1,Diseases of the blood and blood-forming organs,Diseases of the circulatory system,Diseases of the respiratory system,...,No No,No No,No No,No No,No No,No No,No No,No No,No No,No No
70427,359719064,189365864,Other,Male,[60-70),1,1,Diseases of the genitourinary system,"Endocrine, nutritional and metabolic diseases,...",Diseases of the circulatory system,...,No No,No No,No No,No No,No No,No No,No No,No No,No No,No No
70428,338462954,189445127,Caucasian,Female,[80-90),1,1,Diseases of the respiratory system,Diseases of the circulatory system,Diseases of the musculoskeletal system and con...,...,No Change,No No,No No,No Steady,Change No,Change No,Change Steady,No No,No Steady,No Steady
70429,443811536,189481478,Caucasian,Female,[40-50),1,4,Mental disorders,Mental disorders,"Endocrine, nutritional and metabolic diseases,...",...,No No,No Steady,No No,No No,No Steady,No No,No No,Steady No,Steady No,No No
